# 21 — Testing and Debugging

Tests are how you keep the freedom to change code. This lesson covers
pytest's model (plain asserts, fixtures, parametrisation, marks), the
seams that make code testable, `unittest.mock` without over-mocking,
property-based thinking, and the debugging tools — `pdb`, `breakpoint()`,
`logging`, `traceback` — that beat scattering print statements.

`#testing` `#pytest` `#mocking` `#debugging`

<sub>Setup cell — run it once per session so `check()` works.</sub>

In [ ]:
# Wire the grader up. Run this cell first, every session.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents)
              if (p / "tutorial" / "grader.py").exists()), None)
if _root is None:
    raise RuntimeError("run this notebook from inside the tutorial repository")
sys.path.insert(0, str(_root))

from tutorial.grader import check, hint, solution   # noqa: E402

# A notebook kernel already owns an event loop, so asyncio.run() would refuse
# to start another one. Hand those coroutines a private loop on a worker
# thread instead, so the asyncio lessons run here exactly as they do in a
# script. (Outside a notebook this wrapper is a no-op.)
import asyncio as _asyncio                                    # noqa: E402
from concurrent.futures import ThreadPoolExecutor as _Pool    # noqa: E402

_plain_run = _asyncio.run


def _run_anywhere(coro, **kwargs):
    try:
        _asyncio.get_running_loop()
    except RuntimeError:
        return _plain_run(coro, **kwargs)
    with _Pool(1) as pool:
        return pool.submit(_plain_run, coro, **kwargs).result()


_asyncio.run = _run_anywhere

print("grader ready — Python", sys.version.split()[0])

## Why pytest

- plain `assert`, with the expression rewritten so failures show the
  actual values
- fixtures instead of `setUp`/`tearDown` inheritance
- `parametrize` instead of loops inside a test
- a plugin for everything

The single most important habit: **one behaviour per test, named after
the behaviour**. `test_withdraw_rejects_amount_above_balance` tells you
what broke from the failure line alone.

In [ ]:
# A miniature version of pytest's assertion rewriting, to show
# what it buys you.
import ast
import textwrap

def explain_assert(source, namespace):
    tree = ast.parse(textwrap.dedent(source))
    stmt = tree.body[0]
    test = stmt.test
    left = eval(compile(ast.Expression(test.left), "<x>", "eval"), namespace)
    right = eval(compile(ast.Expression(test.comparators[0]), "<x>", "eval"),
                 namespace)
    ok = eval(compile(ast.Expression(test), "<x>", "eval"), namespace)
    return ok, left, right

ns = {"total": 7, "expected": 10}
ok, left, right = explain_assert("assert total == expected", ns)
print("passed:", ok)
print(f"  assert {left} == {right}")
print("  ^ this is why pytest needs no assertEqual family")

pytest rewrites the AST of your test modules at import time,
replacing a bare `assert` with code that evaluates each side
separately and keeps the values around. That is the whole trick,
and this is a miniature of it.

The payoff is that `unittest`'s parallel vocabulary —
`assertEqual`, `assertIn`, `assertAlmostEqual`, and the rest —
becomes unnecessary. You write the comparison Python already has
and still get both operands in the failure message.

It also explains a real pytest gotcha: only test modules and files
registered as plugins are rewritten, so an `assert` inside a
helper imported from elsewhere fails with no detail at all. That
is the reason for `pytest.register_assert_rewrite`.

## Fixtures

A fixture is a function whose return value is injected into any test
that names it as a parameter. Scopes: `function` (default), `class`,
`module`, `package`, `session`.

```python
@pytest.fixture
def db(tmp_path):                 # fixtures can request fixtures
    conn = connect(tmp_path / "test.db")
    yield conn                    # everything after yield is teardown
    conn.close()
```

Built-ins worth knowing: `tmp_path`, `monkeypatch`, `capsys`,
`caplog`, `recwarn`, `request`.

In [ ]:
# Fixtures are just dependency injection. Here is the mechanism,
# implemented in twenty lines, so it stops looking magical.
import inspect

REGISTRY = {}

def fixture(fn):
    REGISTRY[fn.__name__] = fn
    return fn

def run_test(test_fn):
    params = inspect.signature(test_fn).parameters
    finalisers, kwargs = [], {}
    for name in params:
        produced = REGISTRY[name]()
        if inspect.isgenerator(produced):
            kwargs[name] = next(produced)
            finalisers.append(produced)
        else:
            kwargs[name] = produced
    try:
        test_fn(**kwargs)
        return "PASS"
    except AssertionError as exc:
        return f"FAIL: {exc}"
    finally:
        for gen in reversed(finalisers):
            next(gen, None)          # runs the teardown

@fixture
def numbers():
    return [1, 2, 3]

@fixture
def resource():
    print("  setup")
    yield {"open": True}
    print("  teardown")

def test_sum(numbers):
    assert sum(numbers) == 6

def test_uses_resource(resource, numbers):
    assert resource["open"]
    assert len(numbers) == 3

def test_fails(numbers):
    assert sum(numbers) == 7, "sum was 6"

for test in (test_sum, test_uses_resource, test_fails):
    print(f"{test.__name__:22} {run_test(test)}")

Teardown after `yield` runs even when the test fails — that is the
`finally` above. Real pytest adds scoping, caching, parametrised
fixtures and `conftest.py` discovery, but the core is exactly this.

## Parametrise instead of looping

```python
@pytest.mark.parametrize("value,expected", [
    ("", 0), ("a", 1), ("hello world", 2),
])
def test_word_count(value, expected):
    assert word_count(value) == expected
```

Each case is a **separate test**: it gets its own name, its own
failure, and one bad case does not hide the others. A loop inside one
test stops at the first failure and reports one result.

In [ ]:
# Why parametrisation beats a loop: information on failure.
def word_count(text):
    return len(text.split())

cases = [("", 0), ("a", 1), ("hello world", 2), ("  padded  ", 1),
         ("a\nb", 3)]        # deliberately wrong expectation

print("as one looping test:")
try:
    for text, expected in cases:
        assert word_count(text) == expected, f"{text!r}"
except AssertionError as exc:
    print(f"  stopped at the first failure: {exc}")

print("\nas parametrised cases:")
for text, expected in cases:
    got = word_count(text)
    status = "PASS" if got == expected else f"FAIL (got {got})"
    print(f"  word_count({text!r:14}) == {expected}  {status}")
print("\n-> every case reported; you see the whole picture at once")

The looping version stops dead at the first failure, so you learn
about one broken case and nothing about the four behind it. Fix
it, rerun, discover the next one — a serial debugging session
where a parallel one was available.

Parametrised cases are **separate tests**. Each gets its own
result, its own name in the report, and can be rerun on its own
with `pytest -k`. When three of five fail, the pattern among them
is usually the diagnosis.

(The `a\nb` expectation is deliberately wrong: `split()` with no
argument splits on any whitespace, newlines included, so the
answer is 2. Exactly the kind of assumption a table of cases
surfaces and a loop hides.)

### 🧠 Quiz

Why prefer `@pytest.mark.parametrize` over a `for` loop inside one test?

- **a)** It runs faster
- **b)** Each case becomes a separate test: independent failures, individual names, no early exit
- **c)** Loops are not allowed in tests
- **d)** It automatically generates the expected values

<details>
<summary><b>answer</b></summary>

**b** — A loop reports one result and stops at the first failure, hiding how many other cases are broken.

</details>

## Test seams: design for testability

The hard part of testing is rarely the assertion — it is reaching the
code. Make the awkward things injectable:

- **time** — take a `now=time.time` parameter, or inject a clock
- **randomness** — accept a `random.Random(seed)` instance
- **I/O** — accept an open file or a `Path`, not a filename buried
  three layers down
- **network** — put it behind one small function you can replace
- **globals** — pass configuration in

Code that is hard to test is usually code with hidden dependencies.

In [ ]:
import random
import time

# Untestable: hidden dependencies on the clock and the RNG.
def make_token_bad():
    return f"{int(time.time())}-{random.randint(1000, 9999)}"

# Testable: same defaults, but both seams are injectable.
def make_token(now=time.time, rng=random):
    return f"{int(now())}-{rng.randint(1000, 9999)}"

print("production call:", make_token()[:4] + "...")

fixed_clock = lambda: 1_700_000_000
seeded = random.Random(42)
print("deterministic  :", make_token(now=fixed_clock, rng=seeded))
print("reproducible   :", make_token(now=fixed_clock, rng=random.Random(42)))

# And an expiry check becomes trivial to test at any point in time.
def is_expired(issued_at, ttl, now=time.time):
    return now() - issued_at > ttl

print("\nexpired after 61s with ttl 60:",
      is_expired(1000, 60, now=lambda: 1061))
print("not expired after 59s        :",
      is_expired(1000, 60, now=lambda: 1059))

A **seam** is a place where you can change behaviour without
editing the code. `make_token_bad` has none: the clock and the
random source are reached for directly, so testing it means
monkeypatching modules and hoping nothing else in the process
noticed.

The fix costs two default arguments and changes nothing for
callers — production still calls `make_token()`. But the test can
now pin the clock and seed the RNG, and the two identical outputs
show the result: a fully deterministic assertion on a function
whose entire purpose is to be unpredictable.

`is_expired` makes the wider point. Once time is a parameter,
"what happens one second past the boundary" is an ordinary
two-line test rather than something involving `sleep` or a mock.
Dependency injection here is not ceremony; it is the difference
between a test you can write and one you cannot.

## Mocking, and when not to

`unittest.mock` gives you `Mock`, `MagicMock`, `patch`, and
`create_autospec`.

Rules that keep mocks useful:

1. **Patch where it is *used*, not where it is defined.**
   `@patch("myapp.service.requests.get")`, not
   `@patch("requests.get")`.
2. Use `autospec=True` so the mock rejects calls the real object
   would reject.
3. Mock at the **boundary** (network, clock, filesystem), not your own
   internals — mocking your own code tests your mocks.
4. Assert on behaviour, not on every call.

In [ ]:
from unittest.mock import MagicMock, Mock, call, create_autospec

class EmailClient:
    def send(self, to, subject, body): ...
    def close(self): ...

def notify(client, users, subject):
    sent = 0
    for user in users:
        if user.get("email"):
            client.send(user["email"], subject, f"Hi {user['name']}")
            sent += 1
    client.close()
    return sent

client = create_autospec(EmailClient, instance=True)
users = [{"name": "ada", "email": "a@x.com"},
         {"name": "bob", "email": None},
         {"name": "cy", "email": "c@x.com"}]

sent = notify(client, users, "Welcome")
print("returned:", sent)
print("send called", client.send.call_count, "times")
print("first call:", client.send.call_args_list[0])
client.close.assert_called_once()

# autospec catches signature mistakes that a bare Mock would swallow
try:
    client.send("only@one.arg")
except TypeError as exc:
    print("\nautospec caught:", exc)

loose = Mock()
loose.send("only@one.arg")           # a plain Mock accepts anything
print("a bare Mock accepted it:", loose.send.called)

The last four lines are the argument for `create_autospec`. A bare
`Mock` accepts **any** call — wrong arguments, wrong count,
misspelled method names — and returns another Mock, so a test can
pass with total confidence while the real object would have
raised `TypeError`. That is worse than no test: it is a green
check mark over a bug.

`create_autospec` builds the double from the real class, so the
signatures are enforced and typos fail immediately. Make it the
default, and reach for a bare `Mock` only when there is no class
to spec against.

Note also what is being asserted: `call_count`, `call_args_list`
and `assert_called_once` check the **interaction** with the
collaborator, which is the right thing to verify when the useful
output is a side effect rather than a return value.

In [ ]:
from unittest.mock import Mock, patch
import sys
import types

# Building a module so we can demonstrate "patch where it is used".
service = types.ModuleType("service")
exec(
    "import json\n"
    "def fetch(url):\n"
    "    raise RuntimeError('real network call')\n"
    "def get_user(uid):\n"
    "    return fetch(f'/users/{uid}')\n",
    service.__dict__,
)
sys.modules["service"] = service

with patch("service.fetch", return_value={"id": 1, "name": "ada"}) as mocked:
    print("patched result:", service.get_user(1))
    mocked.assert_called_once_with("/users/1")

print("restored after the with block:", end=" ")
try:
    service.get_user(1)
except RuntimeError as exc:
    print(exc)

# side_effect: raise, or return a sequence of values
with patch("service.fetch", side_effect=[{"n": 1}, TimeoutError("slow")]):
    print("\ncall 1:", service.get_user(1))
    try:
        service.get_user(2)
    except TimeoutError as exc:
        print("call 2:", type(exc).__name__, exc)

del sys.modules["service"]

`patch` is a context manager *and* a decorator, and it always
restores the original — even if the test fails. `side_effect`
accepts a list (one value per call), an exception (raised), or a
function (called with the same arguments).

### ❓ Predict the output

Read the cell below and decide what it prints **before** running it. Then run it.

In [ ]:
from unittest.mock import MagicMock

m = MagicMock()
print(bool(m))
print(len(m))
print(str(m.anything.at.all.you.like(1, 2)).split(" id=")[0] + ">")
print(m.some_attr is m.some_attr)
try:
    m.assert_calledd_once()          # note the typo
except AttributeError as exc:
    print("AttributeError:", str(exc)[:46])
print(str(m.was_it_called()).split(" id=")[0] + ">")

<details>
<summary><b>the answer, and why</b></summary>

```
True
0
<MagicMock name='mock.anything.at.all.you.like()'>
True
AttributeError: 'assert_calledd_once' is not a valid assertion
<MagicMock name='mock.was_it_called()'>
```

`MagicMock` pre-configures the dunders (`__len__` → 0, `__bool__`
→ True) and auto-creates any attribute you touch, caching it so
repeated access returns the same child mock.

That convenience is the danger — and mock knows it: names
beginning with `assert`/`assret` raise `AttributeError` precisely
because a misspelled assertion would otherwise pass silently. The
protection stops at that prefix, though: `m.was_it_called()`
happily returns a truthy mock. Use `autospec` or `spec=` and the
whole class of mistake disappears.

</details>

## Property-based testing

Instead of examples, state an **invariant** and let the machine hunt
for counterexamples. `hypothesis` is the library; the idea works with
plain `random` too.

Useful invariant shapes:

- round trip: `decode(encode(x)) == x`
- agreement with a slow, obviously-correct reference implementation
- idempotence: `f(f(x)) == f(x)`
- metamorphic: `sorted(xs + [y])` contains `sorted(xs)` as a
  subsequence

In [ ]:
import random

def encode(items):
    '''Run-length encode a list.'''
    out = []
    for item in items:
        if out and out[-1][0] == item:
            out[-1][1] += 1
        else:
            out.append([item, 1])
    return [(v, n) for v, n in out]

def decode(pairs):
    return [v for v, n in pairs for _ in range(n)]

rng = random.Random(0)
failures = []
for _ in range(400):
    length = rng.randint(0, 12)
    data = [rng.choice("aab") for _ in range(length)]
    if decode(encode(data)) != data:
        failures.append(data)

print("round-trip property held on 400 random inputs:", not failures)

# A deliberately broken variant, to show the property finding it.
def broken_encode(items):
    pairs = encode(items)
    return [(v, n) for v, n in pairs if n > 1]      # drops singletons

counterexample = next(
    (d for d in ([rng.choice("ab") for _ in range(rng.randint(1, 6))]
                 for _ in range(200))
     if decode(broken_encode(d)) != d),
    None,
)
print("counterexample for the broken version:", counterexample)

A **property** is something that must hold for every input rather
than for one example — here, that decoding an encoding returns the
original. One such statement replaces a table of hand-picked
cases, and it covers inputs you would never have thought to write
down.

The counterexample shows the other half of the value: `['b', 'a']`,
two elements, minimal and immediately diagnostic. Property-based
testing tends to find the *small* failing case, which is usually
the one that explains the bug.

In real code you would use Hypothesis rather than this loop: it
generates inputs far more adversarially — empty, huge, unicode,
boundary numbers — and then **shrinks** any failure to the
simplest input that still breaks. Round-trips like this one
(encode/decode, serialise/parse, save/load) are the easiest place
to start, because the property writes itself.

## Debugging beyond `print`

- `breakpoint()` — drops into `pdb` (or whatever `PYTHONBREAKPOINT`
  names). Commands: `n`ext, `s`tep, `c`ontinue, `w`here, `l`ist,
  `p`/`pp` expression, `u`p/`d`own, `b` file:line, `interact`.
- `pdb.post_mortem()` — inspect the frames of an exception *after* it
  happened. `pytest --pdb` does this automatically.
- `logging` with `%`-style lazy formatting, not f-strings, so the
  message is only rendered if the level is enabled.
- `traceback.print_stack()` — how did I get here?
- `sys.settrace` / `sys.monitoring` (3.12) — build your own tracer.

In [ ]:
import logging
import sys
import traceback

logger = logging.getLogger("demo")
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)-7s %(name)s: %(message)s"))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

def risky(n):
    logger.debug("this is never rendered: %s", expensive := "x")
    logger.info("processing %d items", n)
    try:
        return 10 / n
    except ZeroDivisionError:
        logger.exception("division failed")   # includes the traceback
        return None

risky(2)
risky(0)

print()
# post-mortem inspection without a debugger session
try:
    risky(0) or [][1]
except IndexError as exc:
    tb = exc.__traceback__
    while tb.tb_next:
        tb = tb.tb_next
    print("failed in :", tb.tb_frame.f_code.co_name)
    print("locals    :", {k: v for k, v in tb.tb_frame.f_locals.items()
                          if not k.startswith("__")})

`logger.info("got %s", value)` defers the string formatting until
the handler decides it will actually emit. With an f-string you
pay the formatting cost on every call, even at a level you have
turned off — measurable in hot paths and in debug logging.

In [ ]:
import sys

# A tiny call tracer -- the mechanism behind coverage tools,
# debuggers and profilers.
calls = []

def tracer(frame, event, arg):
    if event == "call" and frame.f_code.co_name in {"outer", "inner"}:
        calls.append((frame.f_code.co_name, dict(frame.f_locals)))
    return None

def inner(x): return x * 2
def outer(n): return inner(n) + inner(n + 1)

sys.settrace(tracer)
outer(5)
sys.settrace(None)

for name, local_vars in calls:
    print(f"  called {name}({local_vars})")

`sys.settrace` installs a callback the interpreter invokes on
frame events — `call`, `line`, `return`, `exception`. Coverage
measurement, debuggers and most profilers are this hook plus
bookkeeping; `pdb` is a trace function that happens to prompt you.

Note the frame is live, so `f_locals` gives the arguments as the
callee actually received them — which is why a debugger can show
you variables without any cooperation from the code being
debugged.

The cost is severe: a Python-level callback on every event makes
traced code many times slower, which is why coverage runs are
noticeably sluggish. Returning `None` here, rather than the tracer
itself, declines line-by-line tracing inside each frame and keeps
it to call events only.

### 🧠 Quiz

Why `logger.info('value=%s', x)` rather than `logger.info(f'value={x}')`?

- **a)** f-strings are not supported by logging
- **b)** The %-form defers formatting until the record is actually emitted
- **c)** The %-form is easier to read
- **d)** There is no difference

<details>
<summary><b>answer</b></summary>

**b** — With an f-string the formatting happens before the call, at every call site, even when the level is disabled. It also loses the structured `record.args` that log aggregators can group on.

</details>

---

## ⚒ Exercise — A miniature test runner  
`21-testing/mini-pytest` · difficulty: **brutal**

Build a small test framework with `fixture`, `parametrize`, and
`run_tests` — the core of pytest in about sixty lines.

- `@fixture` registers a function by name; a test declaring a
  parameter of that name receives its value
- a **generator** fixture yields the value and its remainder is
  teardown, run even if the test fails, in reverse order
- fixtures may depend on other fixtures (resolve recursively);
  a dependency cycle raises `ValueError`
- each fixture is created at most once per test, even if two
  fixtures both require it
- `@parametrize("a,b", [(1, 2), (3, 4)])` runs the test once per
  case, filling those parameters; remaining parameters still come
  from fixtures
- `run_tests(tests)` returns a list of result dicts with keys
  `name`, `outcome` (`"pass"`/`"fail"`/`"error"`), and `message`
  — `"fail"` for `AssertionError`, `"error"` for anything else
- a parametrised test's `name` is `f"{fn.__name__}[{case}]"` where
  `case` joins the values with `-`
- a test requesting an unknown fixture is an `"error"`, not a crash

Edit the next cell, run it, then run the check cell below it.

In [ ]:
import inspect

FIXTURES = {}


def fixture(fn):
    ...


def parametrize(argnames, cases):
    ...


def run_tests(tests):
    ...

In [ ]:
check("21-testing/mini-pytest")

<details>
<summary><b>hints</b></summary>

1. Store the parametrise cases on the function itself: `fn._cases = [...]` — that is exactly how pytest's marks work.
2. Resolve fixtures with a recursive function taking a `cache` dict and a `resolving` set; the set is what detects cycles.
3. `inspect.isgeneratorfunction(fn)` tells you whether the fixture needs teardown; call `next(gen)` for the value and `next(gen, None)` to finish it.
4. Run teardowns in a `finally`, in reverse creation order.
5. `argnames` is a comma-separated string — split and strip it, and wrap a single-value case in a tuple so `zip` works.

</details>

<details>
<summary><b>reference solution</b></summary>

```python
import inspect

FIXTURES = {}

def fixture(fn):
    FIXTURES[fn.__name__] = fn
    return fn

def parametrize(argnames, cases):
    names = [n.strip() for n in argnames.split(",")]

    def decorate(fn):
        fn._param_names = names
        fn._param_cases = [
            c if isinstance(c, tuple) else (c,) for c in cases
        ]
        return fn
    return decorate

def _resolve(name, cache, finalisers, resolving):
    if name in cache:
        return cache[name]
    if name in resolving:
        raise ValueError(f"fixture dependency cycle involving {name!r}")
    if name not in FIXTURES:
        raise LookupError(f"unknown fixture {name!r}")

    resolving.add(name)
    fn = FIXTURES[name]
    kwargs = {
        dep: _resolve(dep, cache, finalisers, resolving)
        for dep in inspect.signature(fn).parameters
    }
    resolving.discard(name)

    if inspect.isgeneratorfunction(fn):
        gen = fn(**kwargs)
        value = next(gen)
        finalisers.append(gen)
    else:
        value = fn(**kwargs)
    cache[name] = value
    return value

def _run_one(fn, name, preset):
    cache, finalisers = dict(preset), []
    try:
        kwargs = dict(preset)
        for param in inspect.signature(fn).parameters:
            if param not in kwargs:
                kwargs[param] = _resolve(param, cache, finalisers, set())
        fn(**kwargs)
        return {"name": name, "outcome": "pass", "message": ""}
    except AssertionError as exc:
        return {"name": name, "outcome": "fail", "message": str(exc)}
    except Exception as exc:
        return {"name": name,
                "outcome": "error",
                "message": f"{type(exc).__name__}: {exc}"}
    finally:
        for gen in reversed(finalisers):
            next(gen, None)

def run_tests(tests):
    results = []
    for fn in tests:
        cases = getattr(fn, "_param_cases", None)
        if cases is None:
            results.append(_run_one(fn, fn.__name__, {}))
            continue
        names = fn._param_names
        for case in cases:
            label = "-".join(str(v) for v in case)
            results.append(
                _run_one(fn, f"{fn.__name__}[{label}]", dict(zip(names, case)))
            )
    return results
```

</details>

## Takeaways

- One behaviour per test, named after the behaviour.
- Fixtures are dependency injection; teardown lives after `yield`.
- `parametrize` gives independent, individually-named cases.
- Inject the clock, the RNG, and I/O — untestable code is code with
  hidden dependencies.
- Mock at boundaries with `autospec`; a bare `MagicMock` will happily
  accept a misspelled assertion.

---

### ✅ Lesson complete

Exercises in this lesson: **1**. Check your standing across the whole course with:

```
uv run learn.py list
```